# Exploratory Data Analysis

The canonical v1 workbook supplies revised BOM content only. Its actual single sheet contains no price histories. Retained RM and energy workbooks supply non-overlapping histories and metadata. Outputs are local and Git-ignored. No forecasting models are trained. Full-sample descriptive results are separate from September-origin inputs selected with explicit lag assumptions; November retrieval is not historical availability evidence.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, display

root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/schneider_electric_forecasting").is_dir()
)
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root / "scripts"))

from schneider_electric_forecasting._03_eda import run_eda_report  # noqa: E402

results = run_eda_report()
tables = results["tables"]
out = results["output_directory"]

## Canonical BOM and catalog
Summary/footer rows are preserved separately. Mass shares are not spend weights. Licensed series remain unavailable; generic PPIs are proxies rather than grade-specific market prices.

In [ ]:
display(results["bom"])
display(results["catalog"])

## Overview, availability and missingness
Coverage describes observations in the retrieved snapshot, not publication availability. Monthly calendar gaps, duplicate keys, numeric values and catalog metadata are validated before analysis.

In [ ]:
display(tables["coverage"])
print((out / "quality_checks.json").read_text())
display(Image(filename=str(out / "01_data_availability_heatmap.png")))

## Trends, monthly returns and volatility
Rebased trends use the earliest common valid month. Returns are never filled across gaps. Annualized volatility is sample monthly return standard deviation multiplied by √12; full sample and origin subsets are reported separately.

In [ ]:
display(tables["full_sample_summary"])
display(tables["full_sample_returns"].head())
display(tables["full_sample_volatility"])
display(tables["origin_assumed_volatility"])
for figure in (
    "02_indexed_historical_material_prices",
    "03_pre_origin_annualized_volatility_ranking",
    "04_12_month_rolling_volatility",
):
    display(Image(filename=str(out / f"{figure}.png")))

## Distributions, outliers and seasonality
IQR flags retain all observations. Calendar-month return averages are descriptive and sensitive to structural breaks and sample size.

In [ ]:
display(tables["outliers"].query("is_outlier"))
display(tables["seasonality"])
for figure in ("07_distributions", "08_seasonality"):
    display(Image(filename=str(out / f"{figure}.png")))

## Stationarity, ACF and PACF
ADF uses statsmodels with AIC lag selection and a constant; its null is a unit root. KPSS uses automatic bandwidth and a constant; its null is level stationarity. Tabulated KPSS p-values may be bounds, recorded in the output. Gapped samples are rejected. ACF/PACF use validated library implementations; tests are exploratory and do not validate a forecasting model.

In [ ]:
display(tables["stationarity_levels"])
display(tables["stationarity_returns"])
display(tables["acf_pacf"].head())
display(Image(filename=str(out / "09_acf_pacf.png")))

## Raw material, energy and FX associations
Combined return correlations use pairwise observations and report counts. Duplicate FX E23/S06 is included once. Positive explanatory lags use past observations. Full-sample lag scans do not establish predictive availability, significance or causality.

In [ ]:
display(tables["full_sample_correlations"])
display(tables["full_sample_correlation_counts"])
display(tables["lagged_correlations"])
display(Image(filename=str(out / "05_monthly_return_correlation_heatmap.png")))

## Historical 3-, 6- and 9-month changes
P10/P50/P90 summarize overlapping realized changes, not predictive intervals. Origin-only results exclude post-cutoff target months; full-sample results include retrospective outcomes.

In [ ]:
display(tables["full_sample_historical_horizon_quantiles"])
display(tables["origin_assumed_historical_horizon_quantiles"])
display(Image(filename=str(out / "06_historical_price_risk_by_horizon.png")))

## Updated versus previous EDA and forecasting implications
The September decision is September 30, 2026. Supported source lags recalculate the endpoint; other providers remain excluded from origin inputs. Strict point-in-time readiness requires verified vintages. BOM changes affect product exposures while retained commodity histories remain unchanged. Resolve unmapped brass, unknown composition and the RMAirSeT mass inconsistency before product-cost estimates. Feature engineering can proceed with explicit provenance, lagged inputs, calendar alignment and training-only transformations; models are not trained here.

In [ ]:
from build_reports import build_eda_report

build_eda_report()
print((out / "migration_and_eda_report.md").read_text())